In [ ]:
# ============================================================
# CELL 1 — Imports & settings
# Run once. Edit settings here before starting.
# ============================================================

import numpy as np
import open3d as o3d
import os, sys
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d

# ── File & frame ──────────────────────────────────────────
INPUT_FILE  = r'C:\ROSBAGS VERWIJDER NA BEP\14_40_00\rosbag\rosbag_0.mcap'

# Number of consecutive frames to overlay
N_FRAMES_BEFORE = 25
N_FRAMES_AFTER  = 25
N_FRAMES_OVERLAY = N_FRAMES_BEFORE + N_FRAMES_AFTER + 1  # +1 for center frame

# ── Sensor topics ─────────────────────────────────────────
LIDAR_TOPICS = [
    "/rslidar/M1P_deskewed",
    "/rslidar/helios_R",
    "/rslidar/helios_L",
]

# ── Sensor transforms (from Foxglove TF tree) ─────────────
SENSOR_TRANSFORMS = {
    "/rslidar/M1P_deskewed":  {"translation": [ 0.800,  0.000, 0.876], "rotation_rpy_deg": [0.1, -1.7,  1.7]},
    "/rslidar/helios_R":      {"translation": [-0.743, -0.243, 0.857], "rotation_rpy_deg": [2.4, -2.6, -179.9]},
    "/rslidar/helios_L":      {"translation": [-0.745,  0.191, 0.876], "rotation_rpy_deg": [3.5, -2.1,  179.9]},
}

# ── CSF parameters ────────────────────────────────────────
CLOTH_RESOLUTION = 0.5
SLOPE_SMOOTH     = True
THRESHOLD        = 0.4

# ── Profile parameters ────────────────────────────────────
Y_SLICE       = 0.5    # ±meters from centerline to include
BIN_SIZE      = 0.2    # meters per bin along X
SMOOTH_WINDOW = 10      # bins to smooth over

# ── Output ────────────────────────────────────────────────
OUTPUT_PLOT = r'C:\ROSBAGS VERWIJDER NA BEP\ground_profile.png'

print("Settings loaded.")


In [ ]:
# ============================================================
# CELL 1b — Frame estimator from timestamp
# Reads actual timestamps from the mcap file automatically.
# Run once after Cell 1, then rerun whenever you change timestamp.
# ============================================================

from mcap.reader import make_reader
from mcap_ros2.decoder import DecoderFactory

TARGET_TIMESTAMP = 1777639374.661044896  # <-- paste your Foxglove timestamp here

REFERENCE_TOPIC = "/rslidar/M1P_deskewed"  # topic to use for frame counting

print(f"Scanning timestamps from '{REFERENCE_TOPIC}'...")
print("(This may take a moment...)")

timestamps = []
with open(INPUT_FILE, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(topics=[REFERENCE_TOPIC]):
        # log_time is in nanoseconds — convert to seconds
        timestamps.append(message.log_time / 1e9)

timestamps = np.array(timestamps)
print(f"Found {len(timestamps)} frames")
print(f"Recording start: {timestamps[0]:.3f}")
print(f"Recording end:   {timestamps[-1]:.3f}")
print(f"Duration:        {timestamps[-1] - timestamps[0]:.2f}s")

# Find closest frame to target timestamp
diffs          = np.abs(timestamps - TARGET_TIMESTAMP)
frame_estimate = int(np.argmin(diffs))
closest_time   = timestamps[frame_estimate]
time_error     = abs(closest_time - TARGET_TIMESTAMP)

print(f"\nTarget timestamp:  {TARGET_TIMESTAMP:.3f}")
print(f"Closest timestamp: {closest_time:.3f}  (error: {time_error*1000:.1f}ms)")
print(f"→ Estimated frame: {frame_estimate}")

# Auto-update FRAME_INDEX
FRAME_INDEX = frame_estimate
print(f"\nFRAME_INDEX updated to {FRAME_INDEX} — rerun from Cell 3")

In [ ]:
# ============================================================
# CELL 2 — Helper functions
# Run once.
# ============================================================

def _ros_pc2_to_numpy(msg):
    import numpy as np
    field_map = {f.name: f for f in msg.fields}
    if not all(k in field_map for k in ("x", "y", "z")):
        return None

    # Read the raw bytes as a structured numpy array — no Python loop needed
    dtype_fields = []
    for f in sorted(msg.fields, key=lambda f: f.offset):
        # map ROS field datatypes to numpy
        ros_to_np = {1: 'i1', 2: 'u1', 3: 'i2', 4: 'u2', 5: 'i4', 6: 'u4', 7: 'f4', 8: 'f8'}
        np_type = ros_to_np.get(f.datatype, 'f4')
        dtype_fields.append((f.name, np_type))

    dtype   = np.dtype(dtype_fields)
    n       = msg.width * msg.height
    raw     = np.frombuffer(bytes(msg.data), dtype=np.uint8)
    
    # Extract x, y, z using offsets directly — handles non-packed structs
    x_off = field_map["x"].offset
    y_off = field_map["y"].offset
    z_off = field_map["z"].offset
    step  = msg.point_step

    # Reshape into (n_points, point_step) and slice out x y z
    raw_2d = raw[:n * step].reshape(n, step)
    x = raw_2d[:, x_off:x_off+4].view(np.float32).reshape(-1)
    y = raw_2d[:, y_off:y_off+4].view(np.float32).reshape(-1)
    z = raw_2d[:, z_off:z_off+4].view(np.float32).reshape(-1)

    xyz = np.stack([x, y, z], axis=1)
    return xyz[np.isfinite(xyz).all(axis=1)]

In [ ]:
# ============================================================
# CELL 3 — Load N frames around FRAME_INDEX (before + after)
# SLOW — only rerun if you change FRAME_INDEX or N_FRAMES_*
# ============================================================

from mcap.reader import make_reader
from mcap_ros2.decoder import DecoderFactory

start_frame = FRAME_INDEX - N_FRAMES_BEFORE
end_frame   = FRAME_INDEX + N_FRAMES_AFTER

print(f"Loading frames {start_frame}–{end_frame} ({N_FRAMES_OVERLAY} total) "
      f"from {len(LIDAR_TOPICS)} sensors...")

all_raw_clouds = [dict() for _ in range(N_FRAMES_OVERLAY)]

for topic in LIDAR_TOPICS:
    print(f"  {topic}")
    with open(INPUT_FILE, "rb") as f:
        reader = make_reader(f, decoder_factories=[DecoderFactory()])
        idx = 0
        for schema, channel, message, ros_msg in reader.iter_decoded_messages(topics=[topic]):
            offset = idx - start_frame
            if 0 <= offset < N_FRAMES_OVERLAY:
                pts = _ros_pc2_to_numpy(ros_msg)
                if pts is not None and len(pts) > 0:
                    all_raw_clouds[offset][topic] = pts
            idx += 1
            if idx > end_frame:
                break

print(f"\nLoaded {N_FRAMES_OVERLAY} frame sets ({start_frame} to {end_frame}).")

In [ ]:
# ============================================================
# CELL 4 — Apply transforms & merge each frame
# Rerun if you change SENSOR_TRANSFORMS
# ============================================================

def rpy_deg_to_rotation_matrix(roll_deg, pitch_deg, yaw_deg):
    r, p, y = np.radians(roll_deg), np.radians(pitch_deg), np.radians(yaw_deg)
    Rx = np.array([[1,0,0],[0,np.cos(r),-np.sin(r)],[0,np.sin(r),np.cos(r)]])
    Ry = np.array([[np.cos(p),0,np.sin(p)],[0,1,0],[-np.sin(p),0,np.cos(p)]])
    Rz = np.array([[np.cos(y),-np.sin(y),0],[np.sin(y),np.cos(y),0],[0,0,1]])
    return Rz @ Ry @ Rx

all_merged = []
for i, raw_clouds in enumerate(all_raw_clouds):
    transformed = []
    for topic, pts in raw_clouds.items():
        tf  = SENSOR_TRANSFORMS[topic]
        R   = rpy_deg_to_rotation_matrix(*tf["rotation_rpy_deg"])
        t   = np.array(tf["translation"], dtype=np.float32)
        transformed.append((pts @ R.T).astype(np.float32) + t)
    if transformed:
        merged = np.vstack(transformed)
        all_merged.append(merged)
        print(f"  Frame {FRAME_INDEX+i}: {len(merged):,} points")

print(f"\nMerged {len(all_merged)} frames")

In [ ]:
# ============================================================
# CELL 5 — Run CSF ground filter on each frame
# Rerun if you change CSF parameters
# ============================================================
import CSF

all_ground = []
for i, merged in enumerate(all_merged):
    csf = CSF.CSF()
    csf.params.bSloopSmooth     = SLOPE_SMOOTH
    csf.params.cloth_resolution = CLOTH_RESOLUTION
    csf.params.threshold        = THRESHOLD
    csf.setPointCloud(merged)
    g_idx = CSF.VecInt()
    n_idx = CSF.VecInt()
    csf.do_filtering(g_idx, n_idx)
    ground = merged[np.array(list(g_idx))]
    all_ground.append(ground)
    print(f"  Frame {FRAME_INDEX+i}: {len(ground):,} ground points ({100*len(ground)/len(merged):.1f}%)")

In [ ]:
# ============================================================
# CELL 6 — Slice along centerline & bin each frame by X
# Rerun if you change Y_SLICE or BIN_SIZE
# ============================================================

all_profiles = []  # list of (bin_centers, bin_heights) per frame

for i, ground in enumerate(all_ground):
    mask      = np.abs(ground[:, 1]) <= Y_SLICE
    slice_pts = ground[mask]
    if len(slice_pts) < 10:
        all_profiles.append((np.array([]), np.array([])))
        continue

    z_med = np.median(slice_pts[:, 2])
    z_std = np.std(slice_pts[:, 2])
    slice_pts = slice_pts[np.abs(slice_pts[:, 2] - z_med) < 1.5 * z_std]

    x = -slice_pts[:, 0]
    z =  slice_pts[:, 2]

    x_min = np.floor(x.min() / BIN_SIZE) * BIN_SIZE
    x_max = np.ceil(x.max()  / BIN_SIZE) * BIN_SIZE
    bins  = np.arange(x_min, x_max + BIN_SIZE, BIN_SIZE)

    bc, bh = [], []
    for j in range(len(bins) - 1):
        in_bin = (x >= bins[j]) & (x < bins[j+1])
        if in_bin.sum() >= 3:
            bc.append((bins[j] + bins[j+1]) / 2)
            bh.append(np.median(z[in_bin]))
    all_profiles.append((np.array(bc), np.array(bh)))

print(f"Built profiles for {len(all_profiles)} frames")

In [ ]:
# ============================================================
# CELL 7 — Compute slope & curvature for each frame
# Rerun if you change SMOOTH_WINDOW
# ============================================================

all_results = []  # (bc, z_smooth, slope_smooth, curv_smooth)
for bc, bh in all_profiles:
    if len(bc) < SMOOTH_WINDOW:
        all_results.append(None)
        continue
    z_smooth     = uniform_filter1d(bh, size=SMOOTH_WINDOW)
    dz           = np.gradient(z_smooth, bc)
    slope        = np.degrees(np.arctan(dz))
    curv         = np.gradient(dz, bc)
    slope_smooth = uniform_filter1d(slope, size=SMOOTH_WINDOW)
    curv_smooth  = uniform_filter1d(curv,  size=SMOOTH_WINDOW * 3)
    all_results.append((bc, z_smooth, slope_smooth, curv_smooth))

print(f"Computed {sum(r is not None for r in all_results)} valid profiles")

In [ ]:
# ============================================================
# CELL 8 — Plot all frames overlaid
# Rerun freely to tweak the graph appearance
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle(f"Ground Profile — Frames {start_frame}–{end_frame}  |  Y slice ±{Y_SLICE}m",
             fontsize=14, fontweight='bold')

cmap = plt.cm.viridis
for i, result in enumerate(all_results):
    if result is None:
        continue
    bc, zs, ss, cs = result
    color = cmap(i / max(1, N_FRAMES_OVERLAY - 1))
    label = f"Frame {start_frame + i}"
    axes[0].plot(bc, zs, color=color, linewidth=1.3, alpha=0.8, label=label)
    axes[1].plot(bc, ss, color=color, linewidth=1.3, alpha=0.8, label=label)
    axes[2].plot(bc, cs, color=color, linewidth=1.3, alpha=0.8, label=label)

axes[0].set_ylabel("Height Z (m)")
axes[0].set_title("Ground Height")
axes[1].set_ylabel("Slope (degrees)")
axes[1].set_title("Ground Slope")
axes[2].set_ylabel("Curvature (m⁻¹)")
axes[2].set_title("Vertical Curvature")
axes[2].set_xlabel("Forward distance X (m)")

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    ax.set_xlim(-10, 20)
    ax.legend(fontsize=7, loc='upper left', ncol=2)

plt.tight_layout()
plt.savefig(OUTPUT_PLOT, dpi=150, bbox_inches='tight')
print(f"Saved to: {OUTPUT_PLOT}")
plt.show()

In [ ]:
from mcap.reader import make_reader

count = 0
with open(INPUT_FILE, "rb") as f:
    reader = make_reader(f)
    for schema, channel, message in reader.iter_messages(
            topics=["/odometry/local"]):
        print(f"t={message.log_time/1e9:.3f}  channel={channel.topic}")
        count += 1
        if count >= 3:
            break

print(f"Total messages read: {count}")